# Connectivity Terrain + Settlement Heatmap Inputs

Generates three of the connectivity analysis's raw raster inputs:
`elevation.tif` and `slope.tif` (SRTM via Google Earth Engine) and `settlement_heatmap.tif` (a
local kernel-density estimate over `data/settlements.gpkg`).

`scripts/r/R/grid.R`'s `align_to_grid()` reprojects/resamples every connectivity input onto the
30 m master grid regardless of its native source grid/resolution, so substituting these sources
does not require matching the original files' exact extent -- only reasonable coverage of the
buffered study area.

Requires a `.env` file in the repo root with `EE_PROJECT=<your-gee-cloud-project-id>` and Earth
Engine authenticated on this machine (`earthengine authenticate`) for the elevation/slope section
only -- the settlement-heatmap section has no Earth Engine dependency.

## 1. Setup

In [ ]:
import ee
import eetools
import geemap
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.enums import MergeAlg
from rasterio.features import rasterize
from rasterio.transform import from_origin
from scipy.ndimage import gaussian_filter
from shapely.ops import unary_union

from eetools.io import export_image_list_to_drive
from eetools.vectors import get_sites_geometry, vector_files_to_feature_collection
from eetools.visualization.vis_params import SITES_VIS_PARAMS

In [ ]:
try:
    import config
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "config module not found -- run `uv sync` from the repo root to install this "
        "project (including config.py) into the environment."
    ) from exc

In [ ]:
eetools.initialize()

In [ ]:
# Local aliases for readability -- values are single-sourced in config.py.
CRS = config.PROJECT_CRS
OUTPUT_DIR = config.CONNECTIVITY_INPUT_RASTER_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEM_SOURCE = config.CONNECTIVITY_TERRAIN_DEM_SOURCE
TERRAIN_EXPORT_FOLDER = config.CONNECTIVITY_TERRAIN_EXPORT_FOLDER
TERRAIN_SCALE = config.CONNECTIVITY_TERRAIN_SCALE_M

HEATMAP_RADIUS_M = config.SETTLEMENT_HEATMAP_RADIUS_M
HEATMAP_RESOLUTION_M = config.SETTLEMENT_HEATMAP_RESOLUTION_M

## 2. Study-area boundary & project geometry

In [ ]:
site_files = [(site["path"], site["site_id"], site["site_name"]) for site in config.SITES]
sites_fc = vector_files_to_feature_collection(site_files)

# Buffered union of all sites -- same convention as Objectives 1/2 and the condition composite.
project_geom = get_sites_geometry(sites_fc).buffer(config.STUDY_AREA_BUFFER_M, maxError=10)

## 3. Elevation + slope (SRTM, Google Earth Engine)

`DEM_SOURCE` (`config.CONNECTIVITY_TERRAIN_DEM_SOURCE`) is the void-filled SRTM 30 m global DEM --
the standard free elevation source in Earth Engine, used here in place of the original
hand-produced/QGIS `elevation.tif`/`slope.tif`. Slope is Earth Engine's own `ee.Terrain.slope()`
(degrees), matching what `scripts/r/R/pressure.R`'s `slope_scaled()` expects.

In [ ]:
dem = ee.Image(DEM_SOURCE).select("elevation").clip(project_geom)
slope = ee.Terrain.slope(dem).clip(project_geom)

## 4. Visual QA

In [ ]:
Map = geemap.Map()
Map.centerObject(project_geom, 12)
Map.addLayer(sites_fc.style(**SITES_VIS_PARAMS), {}, "Study sites")
Map.addLayer(
    dem,
    {"min": 1400, "max": 2200, "palette": ["440154", "3b528b", "21908d", "5dc963", "fde725"]},
    "Elevation (m)",
)
Map.addLayer(slope, {"min": 0, "max": 30, "palette": ["ffffcc", "fd8d3c", "800026"]}, "Slope (deg)", False)
Map

## 5. Export terrain rasters to Google Drive

Same build-list-then-start convention as Objectives 1/2 and the condition composite.

In [ ]:
TERRAIN_EXPORT_TASKS = [
    (dem, "elevation", TERRAIN_SCALE),
    (slope, "slope", TERRAIN_SCALE),
]
print(f"{len(TERRAIN_EXPORT_TASKS)} terrain export tasks staged.")

**Manual step** -- review the count above, then run to start this batch.

In [ ]:
started = export_image_list_to_drive(
    TERRAIN_EXPORT_TASKS, aoi=project_geom, folder=TERRAIN_EXPORT_FOLDER, crs=CRS
)
print(f"Started {len(started)} terrain export tasks to Drive folder '{TERRAIN_EXPORT_FOLDER}'.")

**Manual step required after export**: download `elevation.tif` and `slope.tif` from the
`TERRAIN_EXPORT_FOLDER` Drive folder into `config.CONNECTIVITY_INPUT_RASTER_DIR` --
`scripts/r/06_prepare_connectivity_inputs.R` reads them from there.

## 6. Settlement heatmap- KDE

Settlement locations are
rasterized onto a `HEATMAP_RESOLUTION_M` grid covering the buffered site union, then
Gaussian-smoothed with `sigma = HEATMAP_RADIUS_M / HEATMAP_RESOLUTION_M` pixels to build a
continuous density surface. Output units are arbitrary -- `scripts/r/R/pressure.R`'s
`settlement_pressure()` robust-percentile-scales this raster itself, so no normalization is done
here.

In [ ]:
def build_local_project_geometry(sites, crs, buffer_m):
    """Local (geopandas-only) equivalent of the GEE `project_geom` above -- buffered union of
    all site boundaries, reprojected to `crs` -- for grid-building without an Earth Engine
    round-trip."""
    boundaries = [gpd.read_file(site["path"]).to_crs(crs) for site in sites]
    unioned = unary_union([geom for gdf in boundaries for geom in gdf.geometry])
    return unioned.buffer(buffer_m)


local_project_geom = build_local_project_geometry(config.SITES, CRS, config.STUDY_AREA_BUFFER_M)

In [ ]:
def build_raster_grid(geom, resolution_m):
    """Empty grid template covering `geom`'s bounds, extent rounded outward to a clean
    `resolution_m` origin -- same convention as R/grid.R's build_master_grid()."""
    xmin, ymin, xmax, ymax = geom.bounds
    xmin = np.floor(xmin / resolution_m) * resolution_m
    ymax = np.ceil(ymax / resolution_m) * resolution_m
    xmax = np.ceil(xmax / resolution_m) * resolution_m
    ymin = np.floor(ymin / resolution_m) * resolution_m
    width = int((xmax - xmin) / resolution_m)
    height = int((ymax - ymin) / resolution_m)
    transform = from_origin(xmin, ymax, resolution_m, resolution_m)
    return transform, width, height


heatmap_transform, heatmap_width, heatmap_height = build_raster_grid(
    local_project_geom, HEATMAP_RESOLUTION_M
)
print(f"Settlement heatmap grid: {heatmap_width}x{heatmap_height} px @ {HEATMAP_RESOLUTION_M}m")

In [ ]:
settlements_gdf = gpd.read_file(config.CONNECTIVITY_INPUT_PATHS["settlements"]).to_crs(CRS)
settlement_points = settlements_gdf.geometry.centroid  # no-op for point geometries, centroid for polygons

point_counts = rasterize(
    [(geom, 1) for geom in settlement_points],
    out_shape=(heatmap_height, heatmap_width),
    transform=heatmap_transform,
    fill=0,
    merge_alg=MergeAlg.add,
    dtype="float32",
)
print(f"Rasterized {len(settlement_points)} settlement locations; {int(point_counts.sum())} counted on-grid.")

In [ ]:
sigma_px = HEATMAP_RADIUS_M / HEATMAP_RESOLUTION_M
settlement_heatmap = gaussian_filter(point_counts, sigma=sigma_px, mode="constant", cval=0.0)

In [ ]:

heatmap_path = config.CONNECTIVITY_INPUT_PATHS["settlement_heatmap"]
profile = {
    "driver": "GTiff",
    "height": heatmap_height,
    "width": heatmap_width,
    "count": 1,
    "dtype": "float32",
    "crs": CRS,
    "transform": heatmap_transform,
    "compress": "deflate",
}

with rasterio.open(heatmap_path, "w", **profile) as dst:
    dst.write(settlement_heatmap.astype("float32"), 1)

print(f"Wrote settlement heatmap to {heatmap_path}")

## Outputs

All outputs land in `config.CONNECTIVITY_INPUT_RASTER_DIR`:

```text
elevation.tif           (from Drive, after the Section 3 manual download step)
slope.tif                (from Drive, after the Section 3 manual download step)
settlement_heatmap.tif   (written directly by Section 6 -- no Drive round-trip)
```

`scripts/r/06_prepare_connectivity_inputs.R` reads all three via `config.CONNECTIVITY_INPUT_PATHS`.